# Label prediction: results

Can the rules predict a patient's label (steroid response, survival, ...)? This notebook only reads the runs saved by `run_predictions.py`; nothing is rerun here.

In [ ]:
import prediction_search as ps
import prediction_vis as pv
import run_predictions as rp

runs = ps.saved_runs(rp.PREDICTIONS, rp.UNITS)
best = pv.best_per_prediction(runs)

## 1. Best result per question

- **Small pale dots**: every rule run (every rule set, feature, filter and model), spread up and down a little so they don't hide each other.
- **Hollow blue dot**: the typical rule run (median).
- **Big blue dot**: the best rule run.
- **Teal, orange, brown**: the best score with no rules - cell composition, pathological score, clinical score.
- **Pink diamond**: the share of Paneth cells among epithelial cells, Azulay et al.'s simple predictor.
- Rows are sorted by how far the best rule run beats the best of these, per patient.

The best of ~200 runs always looks better than it is. Diagnosis is a control with no reason to be predictable, so its best dot shows how high luck alone reaches.

In [ ]:
pv.plot_best(runs, save="best_per_prediction.pdf")

The setting behind each best rule run:

In [ ]:
best[["prediction", "unit", "best_rules", "best_reference", "best_setting",
      "best_model", "best_auc"]].round(1)

## 2. Which kind of rule carries the signal

A row per rule set and feature, a column per question. Each cell is the best % right of that rule set and feature, over every filter and model; 50% or less stays white.

Read a **row** across: is this kind of rule good for many questions? Read a **column** down: is the signal spread over many kinds of rule (steady) or in one cell (maybe luck)?

In [ ]:
pv.plot_rule_kinds(runs, save="rule_kinds.pdf")

## 3. Are both groups predicted right?

One dot per run: % right of one group (x) against the other (y).

- **Top right**: both groups right - a real prediction.
- **Dashed line**: guessing (50% on average).
- **On an edge**: the model calls almost everyone one group; its score looks fine only because one group is large.

In [ ]:
pv.plot_sides(runs, unit="patient", save="both_groups_patient.pdf")

## 4. The rules behind the best run

The biggest weights of the best logistic run on rules. To the right pushes toward the first group (e.g. Responder), to the left toward the other. Blue = attraction, red = avoidance.

These are what one model leaned on, not proof about any single rule.

In [ ]:
for prediction in rp.PREDICTIONS:
    pv.plot_weights(prediction, unit="patient",
                    save=f"weights_{ps.name_of(prediction)}_patient_plot.pdf")

## 5. Against Azulay et al.

Their only prediction is NRM (death not from relapse), in duodenum, on patients with muscle and more than 50 epithelial cells, scored by AUC. Their model: z-score, the first 2 principal components, linear SVM. They report:

- **AUC 91**: 10 imaging features (cell densities, epithelial fractions, immune phenotypes), chosen as the 10 most correlated with NRM.
- **AUC 76**: the Paneth share alone.

**First check, same data:** their model on our Paneth share should land near their 76. If it does, our setup matches theirs, and the other rows can be read against their 91.

**Read with care:**
- "Best rule run" is the best AUC of ~200 runs, so it looks better than it is.
- They seem to have chosen their 10 features once, on all patients; that also makes their 91 look better than it is.
- We have 35 patients after their filter, they had 33.
- AUC from leaving one patient out runs a little low, for one-column predictors most of all.

In [ ]:
pv.plot_paper(runs, save="against_paper.pdf")
pv.against_paper(runs)[["what", "auc"]].round(1)

In [ ]:
pv.plot_smaller_group(runs, save="smaller_group.pdf")

## 6. Real or luck?

Steroid response without Muscle rules, per patient. The labels are shuffled among the patients and the whole rule search is rerun on them, 100 times; each bar counts the best scores of those searches. The line is the best on the real labels.

p = the share of searches at least as good as the real one. With 100 shuffles it can't go below 0.01. Run `permutation_test.py` first.

In [ ]:
pv.plot_permutation({**rp.CORTICO, "without_muscle_rules": True})

## 7. How many rules the best steroid run keeps

The best steroid run's model and feature (Lasso, mean lift), for each rule set and each cut on how many patients a rule must be found in. One panel per version of the question. A cut that works only in one cell may be luck; the permutation test (section 6) already pays for trying all of them.

In [ ]:
steroid = [ps.name_of({**rp.CORTICO, **version}) for version in rp.VERSIONS]
pv.plot_min_patients(runs, steroid, "Lasso logistic regression", "mean lift",
                     save="min_patients_steroid.pdf")
(runs[runs["prediction"].isin(steroid) & runs["unit"].eq("patient")
      & runs["model"].eq("Lasso logistic regression") & runs["feature"].eq("mean lift")]
 [["prediction", "rule_set", "min_patients", "n_features", "balanced_accuracy", "auc",
   "positive_right", "negative_right"]].round(1))